# Entity Resolution — Stage 1: Preprocessing Runner (Google Colab)

This notebook provides an automated environment for executing **Stage 1 (Preprocessing)** of the large-scale entity-resolution pipeline on Google Colab.

### Stage 1 Responsibilities:
1. **Unicode NFKD Normalization**
2. **Case Standardization** (Unicode-aware lowercase)
3. **Punctuation & Separator Normalization** (converted to spaces to preserve token boundaries)
4. **Whitespace Normalization**
5. **Non-destructive Tokenization** (stored as JSON arrays)
6. **Preservation of Numbers & Alphanumeric Tokens**
7. **Missing-Value Handling** (no `"nan"` / `"null"` artifacts)
8. **Row Preservation & Schema Validation**

--- 
## 1. Environment Setup & Dependencies
Clone the repository and switch to the project root directory before installing dependencies.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Blackdevil-com/Wired-Mortus---Amazon-ML.git"
REPO_NAME = "Wired-Mortus---Amazon-ML"

# Ensure we are inside the repository directory
if not Path("src/preprocessing").exists():
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}

print(f"Current working directory: {Path.cwd()}")

In [ ]:
# Install dependencies for Stage 1
!pip install -r requirements-colab.txt

--- 
## 2. Optional: Mount Google Drive
If your raw datasets (`train_source*.tsv`, `test_source*.tsv`) are stored in Google Drive, run this cell to mount Drive and link them.

In [ ]:
# Optional: Mount Google Drive
USE_GOOGLE_DRIVE = False  # Set to True if using datasets stored in Google Drive

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_data_dir = Path('/content/drive/MyDrive/entity-resolution/data/raw')
    if drive_data_dir.exists():
        print(f"Found raw data directory in Google Drive: {drive_data_dir}")
        !mkdir -p data/raw data/processed
        !cp -rn "/content/drive/MyDrive/entity-resolution/data/raw/"* data/raw/
    else:
        print(f"Google Drive directory not found at {drive_data_dir}. Using local data/raw.")
else:
    print("Using local filesystem for datasets.")

--- 
## 3. Verify Project Structure & Datasets
Inspect the workspace to ensure module directories and raw input files are present.

In [ ]:
raw_dir = Path("data/raw")
proc_dir = Path("data/processed")
proc_dir.mkdir(parents=True, exist_ok=True)

raw_files = sorted(list(raw_dir.glob("*.tsv")))
print(f"Raw dataset files found ({len(raw_files)}):")
for f in raw_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name} ({size_mb:.2f} MB)")

--- 
## 4. Run Unit & Integration Tests
Verify all Unicode normalization, punctuation handling, tokenization, number preservation, and pipeline edge cases before processing.

In [ ]:
!pytest tests/ -v

--- 
## 5. Execute Stage 1 Preprocessing
Run the preprocessing CLI across all available datasets.

In [ ]:
# Execute the Stage 1 batch preprocessing CLI
!python run_preprocessing.py

--- 
## 6. Inspect Preprocessed Outputs & Quality Checks
Display sample records to verify normalized strings, tokens, and numeric preservation.

In [ ]:
import pandas as pd
import json

processed_files = sorted(list(proc_dir.glob("*_preprocessed.tsv")))
print(f"Processed files generated ({len(processed_files)}):")

for pfile in processed_files:
    df = pd.read_csv(pfile, sep="\t", nrows=5, dtype=str, keep_default_na=False)
    print("=" * 70)
    print(f"File: {pfile.name} | Total columns: {len(df.columns)}")
    print("=" * 70)
    cols_to_show = [c for c in ['entity_id', 'business_name', 'business_name_normalized', 'business_name_tokens', 'business_address_normalized', 'row_numbers'] if c in df.columns]
    display(df[cols_to_show])